THE RECOMMENDER

In [1]:
import pandas as pd
import numpy as np
import ast
import difflib

In [2]:
df_1 = pd.read_csv("tmdb_5000_movies.csv")
df_2 = pd.read_csv("tmdb_5000_credits.csv")

df = df_1.merge(df_2, on="title")
df = df[["title", "overview", "genres", "keywords", "cast", "crew"]]
print(df.shape)

(4809, 6)


In [3]:
data = df.dropna().copy()
print(data.shape)

(4806, 6)


In [4]:
data.head()

,title,overview,genres,keywords,cast,crew
0,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [5]:
data["overview"] = data["overview"].apply(lambda x: x.split())

In [6]:
def convert(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i["name"])
    return L

data["genres"] = data["genres"].apply(convert)
data["keywords"] = data["keywords"].apply(convert)

In [7]:
def cast_trim(text):
    L = []
    count = 0
    for i in ast.literal_eval(text):
        if count < 3:
            L.append(i["name"])
            count += 1
        else:
            break
    return L

data["cast"] = data["cast"].apply(cast_trim)

In [8]:
def crew_trim(text):
    L = []
    for i in ast.literal_eval(text):
        if i["job"] == "Director":
            L.append("Director: " + i["name"])
        elif i["job"] == "Original Music Composer":
            L.append("Composer: " + i["name"])
    return L

data["crew"] = data["crew"].apply(crew_trim)

In [9]:
data.head()

,title,overview,genres,keywords,cast,crew
0,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]","[Composer: James Horner, Director: James Cameron]"
1,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley]","[Director: Gore Verbinski, Composer: Hans Zimmer]"
2,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[Daniel Craig, Christoph Waltz, Léa Seydoux]","[Composer: Thomas Newman, Director: Sam Mendes]"
3,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...","[Christian Bale, Michael Caine, Gary Oldman]","[Composer: Hans Zimmer, Director: Christopher ..."
4,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","[Taylor Kitsch, Lynn Collins, Samantha Morton]",[Director: Andrew Stanton]


In [10]:
data["genres"] = data["genres"].apply(lambda x: [i.replace(" ", "") for i in x])
data["keywords"] = data["keywords"].apply(lambda x: [i.replace(" ", "") for i in x])
data["cast"] = data["cast"].apply(lambda x: [i.replace(" ", "") for i in x])
data["crew"] = data["crew"].apply(lambda x: [i.replace(" ", "") for i in x])

In [11]:
data.head()

,title,overview,genres,keywords,cast,crew
0,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]","[Composer:JamesHorner, Director:JamesCameron]"
1,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]","[Director:GoreVerbinski, Composer:HansZimmer]"
2,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, basedonnovel, secretagent, sequel, mi6, ...","[DanielCraig, ChristophWaltz, LéaSeydoux]","[Composer:ThomasNewman, Director:SamMendes]"
3,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dccomics, crimefighter, terrorist, secretiden...","[ChristianBale, MichaelCaine, GaryOldman]","[Composer:HansZimmer, Director:ChristopherNolan]"
4,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, ScienceFiction]","[basedonnovel, mars, medallion, spacetravel, p...","[TaylorKitsch, LynnCollins, SamanthaMorton]",[Director:AndrewStanton]


In [27]:
data["tags"] = ( data["overview"] 
                + data["genres"] * 2
                + data["keywords"] * 3
                + data["cast"] * 2
                + data["crew"] * 4
               )

In [28]:
data["tags"] = data["tags"].apply(lambda x: " ".join(x))

In [29]:
data["tags"] = data["tags"].apply(lambda x: x.lower())

In [30]:
print(data["tags"][3])

following the death of district attorney harvey dent, batman assumes responsibility for dent's crimes to protect the late attorney's reputation and is subsequently hunted by the gotham city police department. eight years later, batman encounters the mysterious selina kyle and the villainous bane, a new terrorist leader who overwhelms gotham's finest. the dark knight resurfaces to protect a city that has branded him an enemy. action crime drama thriller action crime drama thriller dccomics crimefighter terrorist secretidentity burglar hostagedrama timebomb gothamcity vigilante cover-up superhero villainess tragichero terrorism destruction catwoman catburglar imax flood criminalunderworld batman dccomics crimefighter terrorist secretidentity burglar hostagedrama timebomb gothamcity vigilante cover-up superhero villainess tragichero terrorism destruction catwoman catburglar imax flood criminalunderworld batman dccomics crimefighter terrorist secretidentity burglar hostagedrama timebomb go

In [31]:
from sklearn.feature_extraction.text import TfidfVectorizer

toph = TfidfVectorizer(max_features = 5000, stop_words='english')
toph_matrix = toph.fit_transform(data["tags"])

In [32]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(toph_matrix)

In [57]:
def recommender(movie):
    moviess = difflib.get_close_matches(movie, data["title"], n=3, cutoff=0.6)
    movie_name = moviess[0]
    print("Selected Movie: ", movie_name)

    idx = data[data["title"] == movie_name].index[0]

    distance = list(enumerate(similarity[idx]))

    distance = sorted(distance, key=lambda x: x[1], reverse=True)

    recommendations = []

    for i in distance[1:6]:
        recommendations.append(data.iloc[i[0]].title)
    return recommendations

In [58]:
recommender("Aladdin")

Selected Movie:  Aladdin


['Tangled', 'The Princess and the Frog', 'Enchanted', 'Pocahontas', 'Frozen']

BUILDING THE API

In [49]:
from flask import Flask, request, jsonify

In [ ]:
app = Flask(__name__)

@app.route("/")
def home():
    return "Movie Recommender API online!"

@app.route("/recommend", methods=["POST"])
def recommending():
    try:
        movie_data = request.json.get("movie")
        if movie_data is None:
            return jsonify({"error": "Enter name again"}), 400
        
        true_name = difflib.get_close_matches(movie_data, data["title"], n=3, cutoff=0.6)
        if len(true_name) == 0:
            return jsonify({"error": "Movie not found"}), 400
        
        the_true_name = true_name[0]

        return jsonify({
            "Movie Name": the_true_name,
            "Predictions": recommender(movie_data)
        })       

        
            
    except Exception as e:
        return jsonify({
            "error": str(e)
        }), 500

if __name__ == "__main__":
    app.run(debug=True, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [13/May/2026 17:35:47] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [13/May/2026 17:36:17] "POST /recommend HTTP/1.1" 200 -


Selected Movie:  Batman


127.0.0.1 - - [13/May/2026 17:36:32] "POST /recommend HTTP/1.1" 200 -


Selected Movie:  Pocahontas
